In [2]:
# =========================
# Q8 - Sistema de Recomendação
# Produto alvo:
# GPS Garmin Vortex Maré Drift
# =========================

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# =========================
# 1. Carregar dados
# =========================

vendas = pd.read_csv("../data/raw/vendas_2023_2024.csv")
produtos = pd.read_csv("../data/processed/produtos_clean.csv")

# =========================
# 2. Identificar produto alvo
# =========================

produto_nome = "GPS Garmin Vortex Maré Drift"

produto_id = produtos.loc[
    produtos["name"] == produto_nome,
    "code"
].iloc[0]

print("Produto selecionado:", produto_nome)
print("ID:", produto_id)

# =========================
# 3. Criar matriz usuário × produto
# =========================

matriz_usuario_produto = pd.crosstab(
    vendas["id_client"],
    vendas["id_product"]
)

# Converter para binário (presença/ausência)
matriz_usuario_produto = (matriz_usuario_produto > 0).astype(int)

print("\nDimensão matriz:", matriz_usuario_produto.shape)

# =========================
# 4. Similaridade produto × produto
# =========================

matriz_produto_usuario = matriz_usuario_produto.T

similaridade = cosine_similarity(matriz_produto_usuario)

similaridade_df = pd.DataFrame(
    similaridade,
    index=matriz_produto_usuario.index,
    columns=matriz_produto_usuario.index
)

# =========================
# 5. Ranking produtos similares
# =========================

ranking = (
    similaridade_df[produto_id]
    .sort_values(ascending=False)
    .drop(produto_id)
    .head(5)
    .reset_index()
)

ranking.columns = ["product_id", "similaridade"]

# =========================
# 6. Adicionar nome dos produtos
# =========================

ranking = ranking.merge(
    produtos[["code", "name"]],
    left_on="product_id",
    right_on="code",
    how="left"
).drop(columns="code")

print("\nTop 5 produtos similares:\n")
print(ranking)

# =========================
# 7. Produto mais similar
# =========================

produto_top1 = ranking.iloc[0]["product_id"]

print("\nProduto mais similar:", produto_top1)

Produto selecionado: GPS Garmin Vortex Maré Drift
ID: 27



Dimensão matriz: (49, 150)

Top 5 produtos similares:

   product_id  similaridade                                        name
0          94      0.869626            Motor de Popa Volvo Magnum 276HP
1          11      0.868037         GPS Furuno Swift Leviathan Poseidon
2          35      0.853913                          Radar Furuno Swift
3         115      0.850000  Cabo de Nylon Delta Force Magnum Leviathan
4           1      0.850000                 Transponder AIS Maré Magnum

Produto mais similar: 94
